# Fine-tuning QLoRA d'un petit modèle sur le kabyle

**Kabyle LLM Benchmark, phase 2, approche 3.** Ce notebook entraîne un adaptateur LoRA sur un modèle quantifié en 4 bits (QLoRA), à partir des paires français–kabyle **décontaminées** produites par `pipelines/prepare.py`. Il exporte ensuite le modèle au format GGUF pour le servir localement avec Ollama et l'évaluer dans le labo.

## Avant de lancer : coûts et durées estimés

Ce sont des estimations grossières, non mesurées. Hypothèses : modèle de 4 milliards de paramètres en 4 bits, LoRA r=16, environ 60 tokens par exemple.

| Plateforme | GPU | Coût | 60 000 exemples, 1 époque | 185 000 exemples (tout), 1 époque |
|---|---|---|---|---|
| Google Colab (gratuit) | T4 16 Go (non garanti) | 0 € | ≈ 1 à 1,5 h | ≈ 3 à 4 h, au risque d'une coupure de session |
| Kaggle (gratuit, ≈ 30 h GPU/semaine) | T4 ×2 ou P100 | 0 € | ≈ 1 à 1,5 h | ≈ 3 à 4 h |
| RunPod / Vast.ai | RTX 4090 | ≈ 0,34 à 0,90 $/h | ≈ 10 à 20 min | ≈ 30 à 60 min, soit moins de 1 $ |

Recommandation : commencer sur **Kaggle ou Colab gratuit avec 60 000 exemples**. Ne louer un GPU que si le gain mesuré justifie d'entraîner sur tout le corpus.

## Règle anti-contamination

N'utiliser **que** `sft_train.jsonl` et `sft_valid.jsonl`, générés par `pipelines/prepare.py` **après** la saisie du jeu de test. Ce script retire toute phrase qui recoupe un test, et `pipelines/check_contamination.py` le vérifie. Ne jamais charger un fichier de `data/test/` dans ce notebook.

## Étapes

1. Dans le labo (sur le PC) : `docker compose run --rm api python -m pipelines.prepare`, puis `… pipelines.check_contamination`.
2. Activer le GPU : Colab via *Exécution → Modifier le type d'exécution → T4* ; Kaggle via *Settings → Accelerator → GPU*.
3. Exécuter les cellules dans l'ordre. La 1re fois, autoriser Google Drive puis téléverser `sft_train.jsonl` et `sft_valid.jsonl` : ils sont copiés dans *Mon Drive/kab-llm*.
   **Si la session est coupée** : rouvrir le notebook, *Tout exécuter*. Plus de téléversement, et l'entraînement reprend au dernier point de sauvegarde (toutes les 200 étapes, dans Drive).
4. Télécharger le fichier `.gguf` produit et suivre `notebooks/README.md` pour le servir avec Ollama.

In [ ]:
# Unsloth : QLoRA environ 2 fois plus rapide et moins gourmand en mémoire.
# trl (entraînement) est installé explicitement : il manque parfois après « pip install unsloth ».
!pip install -q unsloth trl
import importlib
for lib in ("unsloth", "trl", "datasets", "transformers"):
    print(lib, importlib.import_module(lib).__version__)


In [ ]:
# ── Configuration ──────────────────────────────────────────
# Modèle à entraîner : "qwen3.5-4b" ou "ministral3-8b" (mêmes données, mêmes réglages)
MODEL_CHOICE = "ministral3-8b"
PRESETS = {
    # même modèle que « qwen3.5-4b » dans le labo
    "qwen3.5-4b": dict(base="unsloth/Qwen3.5-4B", output="kab-qwen3.5-4b", batch=8, accum=2,
                       user="<|im_start|>user\n", assistant="<|im_start|>assistant\n"),
    # même modèle que « ministral3-8b » (ollama ministral-3:8b) dans le labo ; lot de 4 x 4
    # au lieu de 8 x 2 pour tenir sur un T4 : même taille de lot effective (16), mêmes étapes
    "ministral3-8b": dict(base="unsloth/Ministral-3-8B-Instruct-2512", output="kab-ministral3-8b",
                          batch=4, accum=4, user="[INST]", assistant="[/INST]"),
}
P = PRESETS[MODEL_CHOICE]
BASE_MODEL, OUTPUT_NAME = P["base"], P["output"]
MAX_SEQ_LENGTH = 512        # les paires Tatoeba sont courtes
MAX_TRAIN_EXAMPLES = 60_000 # None = tout le corpus (≈ 185 000 exemples)
EPOCHS = 1
LEARNING_RATE = 2e-4
LORA_RANK = 16
SEED = 13

In [ ]:
# ── Données et sauvegardes dans Google Drive ───────────────
# Colab gratuit coupe les sessions : tout ce qui compte (données, points de sauvegarde,
# adaptateur) est rangé dans Mon Drive/kab-llm pour reprendre là où l'entraînement s'est arrêté.
import os, shutil

DATA_DIR = "."
SAVE_DIR = "."
# Colab a un dossier /kaggle et Kaggle a le module google.colab : on se fie à des
# marqueurs propres à chacun (variable d'environnement Kaggle, fichier système Colab)
IN_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))
IN_COLAB = not IN_KAGGLE and os.path.exists("/var/colab/hostname")
print("Kaggle" if IN_KAGGLE else "Colab" if IN_COLAB else "autre machine")
if IN_KAGGLE:
    # Sur Kaggle : ajouter un « Dataset » contenant les deux fichiers
    candidates = [r for r, _, files in os.walk("/kaggle/input") if "sft_train.jsonl" in files]
    DATA_DIR = candidates[0] if candidates else "."
    SAVE_DIR = "/kaggle/working"
elif IN_COLAB:
    from google.colab import drive, files
    drive.mount("/content/drive")
    SAVE_DIR = DATA_DIR = "/content/drive/MyDrive/kab-llm"
    os.makedirs(SAVE_DIR, exist_ok=True)
    if not os.path.exists(f"{DATA_DIR}/sft_train.jsonl"):
        if not os.path.exists("sft_train.jsonl"):  # déjà téléversés dans cette session ?
            files.upload()  # 1re fois seulement : choisir sft_train.jsonl et sft_valid.jsonl
        for name in ("sft_train.jsonl", "sft_valid.jsonl"):
            shutil.copy(name, DATA_DIR)

TRAIN_FILE = os.path.join(DATA_DIR, "sft_train.jsonl")
VALID_FILE = os.path.join(DATA_DIR, "sft_valid.jsonl")
# Qwen a été entraîné avec « checkpoints » : nom gardé pour pouvoir le reprendre
CKPT_NAME = "checkpoints" if MODEL_CHOICE == "qwen3.5-4b" else f"checkpoints-{OUTPUT_NAME}"
CHECKPOINT_DIR = os.path.join(SAVE_DIR, CKPT_NAME)
if IN_KAGGLE and not os.path.isdir(CHECKPOINT_DIR):
    # Reprise sur Kaggle : ajouter en « Input » la sortie de la version précédente de ce notebook.
    # Son dernier point de sauvegarde est recopié ici, et l'entraînement repart de là.
    found = [r for r, _, _ in os.walk("/kaggle/input") if os.path.basename(r).startswith("checkpoint-")
             and os.path.basename(os.path.dirname(r)) == CKPT_NAME  # jamais ceux d'un autre modèle
             and os.path.exists(os.path.join(r, "trainer_state.json"))]
    if found:
        last = max(found, key=lambda p: int(p.rsplit("-", 1)[1]))
        shutil.copytree(last, os.path.join(CHECKPOINT_DIR, os.path.basename(last)))
        print("checkpoint précédent repris :", last)
assert os.path.exists(TRAIN_FILE) and os.path.exists(VALID_FILE), "fichiers d'entraînement introuvables"
print(TRAIN_FILE, VALID_FILE, "| points de sauvegarde :", CHECKPOINT_DIR)


In [ ]:
# ── Modèle de base en 4 bits + adaptateur LoRA ─────────────
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)
model.print_trainable_parameters()

In [ ]:
# ── Mise en forme avec le gabarit de conversation du modèle ─
from datasets import load_dataset
import re

ds = load_dataset("json", data_files={"train": TRAIN_FILE, "valid": VALID_FILE})
if MAX_TRAIN_EXAMPLES:
    ds["train"] = ds["train"].shuffle(seed=SEED).select(range(min(MAX_TRAIN_EXAMPLES, len(ds["train"]))))

# Ministral 3 insère un long message système par défaut : avec lui, les exemples dépassent
# MAX_SEQ_LENGTH, la réponse est tronquée et la perte ne porte plus sur rien. On le retire
# (sans effet sur Qwen, dont le gabarit n'a pas ce bloc).
SYSTEM_BLOCK = re.compile(r"\[SYSTEM_PROMPT\].*?\[/SYSTEM_PROMPT\]", re.S)

def to_text(batch):
    # enable_thinking=False : réponses directes, comme dans le labo (think: false)
    return {"text": [SYSTEM_BLOCK.sub("", tokenizer.apply_chat_template(m, tokenize=False, enable_thinking=False))
                     for m in batch["messages"]]}

ds = ds.map(to_text, batched=True, remove_columns=ds["train"].column_names)
print(ds)
print(ds["train"][0]["text"])

# Contrôle avant d'entraîner : aucun exemple ne doit dépasser MAX_SEQ_LENGTH
tok = getattr(tokenizer, "tokenizer", tokenizer)  # Qwen 3.5 / Ministral 3 : processeur → tokenizer texte
lengths = [len(tok(t)["input_ids"]) for t in ds["train"].select(range(500))["text"]]
print(f"longueur en tokens (500 exemples) : max {max(lengths)} | moyenne {sum(lengths) // len(lengths)}")
assert max(lengths) < MAX_SEQ_LENGTH, "exemples trop longs : la réponse serait tronquée, voir le texte affiché ci-dessus"

In [ ]:
# ── Entraînement ───────────────────────────────────────────
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=ds["train"],
    eval_dataset=ds["valid"],
    args=SFTConfig(
        dataset_text_field="text",
        max_length=MAX_SEQ_LENGTH,  # nom actuel dans TRL (anciennement max_seq_length)
        per_device_train_batch_size=P["batch"],
        gradient_accumulation_steps=P["accum"],
        num_train_epochs=EPOCHS,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="linear",
        warmup_ratio=0.03,
        optim="adamw_8bit",
        logging_steps=25,
        eval_strategy="steps",
        eval_steps=1000,
        save_steps=200,         # ≈ toutes les 5 min : une coupure ne perd presque rien
        save_total_limit=2,
        output_dir=CHECKPOINT_DIR,  # dans Google Drive : survit à la fin de session
        seed=SEED,
        report_to="none",
    ),
)
# La perte ne porte que sur les réponses (pas sur les consignes)
trainer = train_on_responses_only(
    trainer,
    instruction_part=P["user"],
    response_part=P["assistant"],
)

# Reprise automatique : si une session précédente a été coupée, on repart du dernier point de sauvegarde
import glob
resume = bool(glob.glob(os.path.join(CHECKPOINT_DIR, "checkpoint-*")))
print("Reprise du dernier point de sauvegarde" if resume else "Nouvel entraînement")
stats = trainer.train(resume_from_checkpoint=resume)
print(stats)

In [ ]:
# ── Vérification rapide sur le jeu de VALIDATION (jamais le jeu de test) ──
FastLanguageModel.for_inference(model)
import json

with open(VALID_FILE, encoding="utf-8") as f:
    samples = [json.loads(line) for _, line in zip(range(5), f)]
# Qwen 3.5 et Ministral 3 sont multimodaux : le « tokenizer » est un processeur, le texte doit être passé par text=.
# La vérification ne doit jamais empêcher l'export qui suit.
try:
    for s in samples:
        prompt = tokenizer.apply_chat_template(s["messages"][:1], tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
        inputs = tokenizer(text=prompt, return_tensors="pt").to("cuda")
        out = model.generate(**inputs, max_new_tokens=80, do_sample=False)
        answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        print("CONSIGNE :", s["messages"][0]["content"])
        print("MODÈLE   :", answer.strip())
        print("ATTENDU  :", s["messages"][1]["content"], "\n")
except Exception as exc:
    print("Vérification ignorée :", exc)


In [ ]:
# ── Export : adaptateur LoRA + modèle GGUF (Q4_K_M) pour Ollama ──
model.save_pretrained(f"{SAVE_DIR}/{OUTPUT_NAME}-lora")  # adaptateur gardé dans Drive
tokenizer.save_pretrained(f"{SAVE_DIR}/{OUTPUT_NAME}-lora")
model.save_pretrained_gguf(OUTPUT_NAME, tokenizer, quantization_method="q4_k_m")

import glob
print(glob.glob(f"{OUTPUT_NAME}*/*.gguf") + glob.glob(f"{OUTPUT_NAME}*.gguf"))
print("Télécharger le .gguf (et le Modelfile s'il a été généré), puis suivre notebooks/README.md")

In [ ]:
# ── Copie du .gguf dans Google Drive (pour notebooks/03_ollama_colab) ──
# Évite de télécharger ~3 Go sur le PC : le notebook 03 le reprend depuis Drive.
import glob, os, shutil
ggufs = glob.glob(f"{OUTPUT_NAME}*/*.gguf") + glob.glob(f"{OUTPUT_NAME}*.gguf")
if not IN_COLAB:
    # Kaggle : les fichiers restent dans l'onglet « Output » de la version
    print("Pas sur Colab : récupérer ces fichiers dans l'onglet Output :", ggufs)
else:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/kab-llm"
    os.makedirs(dest, exist_ok=True)
    for f in ggufs:
        shutil.copy(f, dest)
        print("copié :", f"kab-llm/{os.path.basename(f)}", "← valeur de GGUF_ON_DRIVE dans le notebook 03")
